In [1]:
import pandas as pd
import re
import os

In [2]:
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator"

In [3]:
# ── Derived input paths ───────────────────────────────────────────────────────
PUBCHEM_PKL_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.paraquet")
PUBCHEM_DB_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
PUBCHEM_SYN_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
ENS2NCBI_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
OMIM_DO_PATH       = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/OMIMinDO.tsv")
MESH2DOID_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv")
MEDGEN_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MeSH/MedGenIDMappings.txt")
MEDGEN_HPO_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MeSH/MedGen_HPO_Mapping.txt")
MESH_COMB_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_Combined.csv")
MESH_SUPP_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/Mesh_Supplementary_Info.csv")
DO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
GO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_GO_ID_NAME.csv")
UBERON_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv")
HPO_PATH           = os.path.join(BASE_PATH, "databases_for_mapping/hpo/HPO.csv")
REACTOME_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/reactome/ReactomePathways.txt")
UNIPROT_REC_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/Uniprot_ID_Recname.csv")
CL_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/CO/cl_id_mapping.csv")

# Per-relation intermediate CSVs are read from this folder
PROCESSED_TARKG    = os.path.join(BASE_PATH, "tarkg/Processed_TARKG/")

os.makedirs(OUT_PATH, exist_ok=True)

# Standard schema column order
DESIRED_COLS = [
    "Head", "Relation", "Tail", "Head_type", "Relation_type", "Tail_type", "Source",
    "KG_Source", "Head_Alt_ID", "Tail_DO_Alt_ids",
    "Head_IUPAC", "Tail_IUPAC", "Head_Smiles", "Head_SMILES_name", "Head_detail_name", 
    "Tail_Smiles", "Tail_SMILES_name", "Tail_detail_name",
    "Head_ID_IS", "Tail_ID_IS", "Kg_index", "Kg", "Change"
]

print("Paths configured.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator


In [4]:
def select_cols(df):
    """Select only columns that exist in df from DESIRED_COLS."""
    return df[[c for c in DESIRED_COLS if c in df.columns]]


def save(df, filename):
    """Save to OUT_PATH and print confirmation."""
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=None)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


In [5]:
# ── 2d. ENSEMBL <-> NCBI gene crosswalk ──────────────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [6]:
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID']   = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict         = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict       = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
NCBI_Synbol_GENEname_dict      = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
# String-keyed version needed for TARKG gene IDs which come as strings
NCBI_ALL_GENEIDname_dict_str   = {str(k): v for k, v in NCBI_ALL_GENEIDname_dict.items()}
print(f"NCBI Human genes: {len(NCBI_Synbol_GENEname_dict):,}")
NCBI_ALL_GENE

NCBI Human genes: 193,352


,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type,ENSEMBLE_ID
0,9606,1,A1BG,-,A1B|ABG|GAB|HYST2477,MIM:138670|HGNC:HGNC:5|Ensembl:ENSG00000121410...,19,19q13.43,alpha-1-B glycoprotein,protein-coding,A1BG,alpha-1-B glycoprotein,O,alpha-1B-glycoprotein|HEL-S-163pA|epididymis s...,20250208,-,ENSG00000121410
1,9606,2,A2M,-,A2MD|CPAMD5|FWP007|S863-7,MIM:103950|HGNC:HGNC:7|Ensembl:ENSG00000175899...,12,12p13.31,alpha-2-macroglobulin,protein-coding,A2M,alpha-2-macroglobulin,O,alpha-2-macroglobulin|C3 and PZP-like alpha-2-...,20250208,-,ENSG00000175899
2,9606,3,A2MP1,-,A2MP,HGNC:HGNC:8|Ensembl:ENSG00000291190|AllianceGe...,12,12p13.31,alpha-2-macroglobulin pseudogene 1,pseudo,A2MP1,alpha-2-macroglobulin pseudogene 1,O,pregnancy-zone protein pseudogene,20241210,-,ENSG00000291190
3,9606,9,NAT1,-,AAC1|MNAT|NAT-1|NATI,MIM:108345|HGNC:HGNC:7645|Ensembl:ENSG00000171...,8,8p22,N-acetyltransferase 1,protein-coding,NAT1,N-acetyltransferase 1,O,arylamine N-acetyltransferase 1|N-acetyltransf...,20250208,-,ENSG00000171428
4,9606,10,NAT2,-,AAC2|NAT-2|PNAT,MIM:612182|HGNC:HGNC:7646|Ensembl:ENSG00000156...,8,8p22,N-acetyltransferase 2,protein-coding,NAT2,N-acetyltransferase 2,O,arylamine N-acetyltransferase 2|N-acetyltransf...,20250208,-,ENSG00000156006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193500,741158,8923215,trnD,-,-,-,MT,-,tRNA-Asp,tRNA,-,-,-,-,20200909,-,NaN
193501,741158,8923216,trnP,-,-,-,MT,-,tRNA-Pro,tRNA,-,-,-,-,20200909,-,NaN
193502,741158,8923217,trnA,-,-,-,MT,-,tRNA-Ala,tRNA,-,-,-,-,20200909,-,NaN
193503,741158,8923218,COX1,-,-,-,MT,-,cytochrome c oxidase subunit I,protein-coding,-,-,-,cytochrome c oxidase subunit I,20230818,-,NaN


In [7]:
NCBI_Synbol_GENEname_dict

{'A1BG': 'alpha-1-B glycoprotein',
 'A2M': 'alpha-2-macroglobulin',
 'A2MP1': 'alpha-2-macroglobulin pseudogene 1',
 'NAT1': 'N-acetyltransferase 1',
 'NAT2': 'N-acetyltransferase 2',
 'NATP': 'N-acetyltransferase pseudogene',
 'SERPINA3': 'serpin family A member 3',
 'AADAC': 'arylacetamide deacetylase',
 'AAMP': 'angio associated migratory cell protein',
 'AANAT': 'aralkylamine N-acetyltransferase',
 'AARS1': 'alanyl-tRNA synthetase 1',
 'AAVS1': 'adeno-associated virus integration site 1',
 'ABAT': '4-aminobutyrate aminotransferase',
 'ABCA1': 'ATP binding cassette subfamily A member 1',
 'ABCA2': 'ATP binding cassette subfamily A member 2',
 'ABCA3': 'ATP binding cassette subfamily A member 3',
 'ABCB7': 'ATP binding cassette subfamily B member 7',
 'ABCF1': 'ATP binding cassette subfamily F member 1',
 'ABCA4': 'ATP binding cassette subfamily A member 4',
 'ABL1': 'ABL proto-oncogene 1, non-receptor tyrosine kinase',
 'AOC1': 'amine oxidase copper containing 1',
 'ABL2': 'ABL prot

In [8]:
# ── 2a. PubChem CID -> IUPAC name and SMILES ──────────────────────────────────
Pubchem = pd.read_parquet(PUBCHEM_PKL_PATH)
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
del Pubchem
print("PubChem CID dicts loaded")

PubChem CID dicts loaded


In [9]:
# ── 2. UBERON anatomy, HPO phenotype, Reactome pathways, UniProt ────────────
UBERON = pd.read_csv(UBERON_PATH)
UBERON_dict = dict(zip(UBERON['UBERON ID'], UBERON['Name']))

In [10]:

# ── 2. UBERON anatomy, HPO phenotype, Reactome pathways, UniProt ────────────
CellOnto = pd.read_csv(CL_PATH)
CellOnto_dict = dict(zip(CellOnto['CL_ID'], CellOnto['cell_type_name']))


In [11]:
# ── 2. GO terms + secondary ID resolution ────────────────────────────────────
GO = pd.read_csv(GO_PATH)
GO['namespace'] = GO['namespace'].replace({
    'biological_process': 'Biological Process',
    'molecular_function': 'Molecular Function',
    'cellular_component': 'Cellular Component'
})
GO_dict           = dict(zip(GO['id'], GO['name']))
GO_namespace_dict = dict(zip(GO['id'], GO['namespace']))

# Secondary GO IDs -> primary GO ID (for resolving retired identifiers)
GO_SecID = GO[['id','name','namespace','alt_id']].dropna(subset=['alt_id']).copy()
GO_SecID['alt_id'] = GO_SecID['alt_id'].str.split(r'\s*\|\s*')
GO_SecID = GO_SecID.explode('alt_id')
GO_SecID_2_prim_ID_dict = dict(zip(GO_SecID['alt_id'], GO_SecID['id']))
print(f"GO terms: {len(GO_dict):,} | GO secondary IDs: {len(GO_SecID_2_prim_ID_dict):,}")

GO terms: 47,995 | GO secondary IDs: 3,646


In [12]:
# ── 2b. PubChem crosswalk: DrugBank/ChEBI -> PubChem CID ─────────────────────
pubchem = pd.read_csv(PUBCHEM_DB_PATH)
pubchem
# pubchem_DB    = pubchem[~pubchem['DRUGBANK_ID'].isna()]
pubchem_CHEBI = pubchem[~pubchem['CHEBI_ID'].isna()]
# DB2Pub_dict   = dict(zip(pubchem_DB['ID'],          pubchem_DB['DRUGBANK_ID']))   # CID -> DB ID
# DB2PC_dict    = dict(zip(pubchem_DB['DRUGBANK_ID'],  pubchem_DB['ID']))            # DB ID -> CID
Chebi2PC_dict = dict(zip(pubchem_CHEBI['CHEBI_ID'], pubchem_CHEBI['ID']))          # ChEBI -> CID
# del pubchem
# print(f"DrugBank->PubChem: {len(DB2PC_dict):,} | ChEBI->PubChem: {len(Chebi2PC_dict):,}")
# Chebi2PC_dict

/tmp/ipykernel_2057361/4025516582.py:2: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem = pd.read_csv(PUBCHEM_DB_PATH)


In [13]:
# ── 2j. Disease Ontology (DO) ─────────────────────────────────────────────────
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO entries: {len(DOID_disname_DOID_dict):,}")

DO entries: 11,787


In [14]:
phenotype = pd.read_csv(HPO_PATH)
phenotype = phenotype[phenotype['id'].str.startswith('HP')]
phenotype_dict        = dict(zip(phenotype['name'], phenotype['id']))  # name -> HPO ID
HPOphenotype_name_dict= dict(zip(phenotype['id'],   phenotype['name']))# HPO ID -> name

In [89]:
# HPOphenotype_name_dict

# Prepre processing 

In [37]:
import pandas as pd

df = pd.read_csv(
    "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/pheknowlator/PheKnowLator_v1_Full_BioKG_NoDisjointness_Closed_ELK_Triples_Labels.txt",
    sep="\t", header=None, names=["subject", "object"]
)

def parse_uri(uri):
    uri = str(uri)
    if "obolibrary.org/obo/" in uri:
        raw = uri.split("/obo/")[-1]
        prefix = raw.split("_")[0]
        return raw.replace("_", ":"), prefix
    elif "uniprot.org/geneid/" in uri:
        return "NCBI:" + uri.split("/geneid/")[-1], "Gene"
    elif "reactome.org" in uri or "identifiers.org/reactome" in uri:
        raw = uri.split("/")[-1]
        return raw, "R-HSA"
    else:
        raw = uri.split("/")[-1].split("#")[-1]
        if raw.endswith(".owl"):
            return raw, "Other"
        if raw.startswith("R-HSA"):
            return raw, "R-HSA"
        prefix = raw.split("_")[0].split(":")[0]
        return raw, prefix

prefix_to_type = {
    "DOID":     "Disease",
    "HP":       "Phenotype",
    "GO":       "GO_Term",
    "CHEBI":    "Chemical",
    "Gene":     "Gene",
    "UBERON":   "Anatomy",
    "CL":       "CellType",
    "PR":       "Protein",
    "PRO":      "Protein",
    "R-HSA":    "Pathway",
    "PATO":     "Phenotype_Quality",
    "NCBITaxon":"Species",
    "SO":       "SequenceFeature",
    "FMA":      "Anatomy",
    "HsapDv":   "DevStage",
    "NBO":      "Behavior",
    "CARO":     "Anatomy",
    # Ontology metadata — drop
    "IAO":      "Other",
    "VO":       "Other",
    "RO":       "Other",
    "BFO":      "Other",
    "OBI":      "Other",
    "MPATH":    "Other",
    "OAE":      "Other",
    "OGG":      "Other",
    "BSPO":     "Other",
    "OGMS":     "Other",
    "UBPROP":   "Other",
    "IDO":      "Other",
    "NBO":      "Other",
    "Unknown":  "Other",
}

def get_type(prefix):
    return prefix_to_type.get(prefix, "Other")

df["subject_id"], df["subject_prefix"] = zip(*df["subject"].map(parse_uri))
df["object_id"],  df["object_prefix"]  = zip(*df["object"].map(parse_uri))
df["subject_type"] = df["subject_prefix"].map(get_type)
df["object_type"]  = df["object_prefix"].map(get_type)

type_pair_to_relation = {
    ("Disease",   "GO_Term"):         "disease_associated_with_go",
    ("Disease",   "Pathway"):         "disease_associated_with_pathway",
    ("Disease",   "Phenotype"):       "disease_has_phenotype",
    ("Chemical",  "Disease"):         "chemical_associated_with_disease",
    ("Chemical",  "Gene"):            "chemical_interacts_with_gene",
    ("Chemical",  "Pathway"):         "chemical_associated_with_pathway",
    ("GO_Term",   "GO_Term"):         "go_subclass_of",
    ("GO_Term",   "Gene"):            "go_associated_with_gene",
    ("GO_Term",   "Pathway"):         "go_associated_with_pathway",
    ("Gene",      "Gene"):            "gene_interacts_with_gene",
    ("Gene",      "Pathway"):         "gene_participates_in_pathway",
    ("Gene",      "Disease"):         "gene_associated_with_disease",
    ("Phenotype", "Gene"):            "phenotype_associated_with_gene",
    ("Phenotype", "Disease"):         "phenotype_associated_with_disease",
    ("Protein",   "Protein"):         "protein_interacts_with_protein",
    ("Anatomy",   "Gene"):            "anatomy_expresses_gene",
    ("Anatomy",   "Anatomy"):         "anatomy_subclass_of",
    ("CellType",  "Anatomy"):         "celltype_part_of_anatomy",
    ("Species",   "Gene"):            "species_has_gene",
}

df["relation"] = df.apply(
    lambda r: type_pair_to_relation.get(
        (r["subject_type"], r["object_type"]), "related_to"
    ), axis=1
)

edges = df[[
    "subject_id", "relation", "object_id",
    "subject_type", "object_type"
]].rename(columns={
    "subject_id":   "Head",
    "object_id":    "Tail",
    "subject_type": "Head_type",
    "object_type":  "Tail_type",
    "relation":  "Relation"
})
edges["source"] = "PheKnowLator"

keep_types = {
    "Disease", "Phenotype", "GO_Term", "Chemical",
    "Gene", "Anatomy", "CellType", "Protein", "Pathway",
    "Phenotype_Quality", "Species", "SequenceFeature", "DevStage"
}

edges = edges[
    edges["Head_type"].isin(keep_types) &
    edges["Tail_type"].isin(keep_types)
].reset_index(drop=True)

print(edges.shape)
print(edges[["Head_type", "Tail_type", "Relation"]].value_counts().head(30))
edges

(4210938, 6)
Head_type          Tail_type          Relation                        
Disease            GO_Term            disease_associated_with_go          1429336
Chemical           Disease            chemical_associated_with_disease     756419
GO_Term            Gene               go_associated_with_gene              612393
Gene               Gene               gene_interacts_with_gene             411628
Chemical           Gene               chemical_interacts_with_gene         243902
                   Chemical           related_to                           200531
Phenotype          Gene               phenotype_associated_with_gene       150653
Gene               Pathway            gene_participates_in_pathway         109180
GO_Term            GO_Term            go_subclass_of                        95169
Disease            Phenotype          disease_has_phenotype                 64105
GO_Term            Pathway            go_associated_with_pathway            53067
Phenotype     

,Head,Relation,Tail,Head_type,Tail_type,source
0,DOID:5041,disease_associated_with_go,GO:0002537,Disease,GO_Term,PheKnowLator
1,DOID:1588,disease_associated_with_go,GO:0070269,Disease,GO_Term,PheKnowLator
2,GO:0035377,go_subclass_of,GO:0006833,GO_Term,GO_Term,PheKnowLator
3,CHEBI:35455,chemical_associated_with_disease,DOID:2799,Chemical,Disease,PheKnowLator
4,CHEBI:15930,chemical_interacts_with_gene,NCBI:10640,Chemical,Gene,PheKnowLator
...,...,...,...,...,...,...
4210933,DOID:8442,disease_associated_with_go,GO:0006749,Disease,GO_Term,PheKnowLator
4210934,GO:0030553,go_subclass_of,GO:0030553,GO_Term,GO_Term,PheKnowLator
4210935,DOID:10933,disease_associated_with_go,GO:1903188,Disease,GO_Term,PheKnowLator
4210936,DOID:12336,disease_associated_with_go,GO:0046500,Disease,GO_Term,PheKnowLator


In [38]:
edges['Relation'] = edges['Head_type'] + '_' + edges['Tail_type']

In [39]:
set(edges['Head_type']), set(edges['Tail_type']), set(edges['Relation'])

({'Anatomy',
  'CellType',
  'Chemical',
  'DevStage',
  'Disease',
  'GO_Term',
  'Gene',
  'Phenotype',
  'Phenotype_Quality',
  'Protein',
  'SequenceFeature',
  'Species'},
 {'Anatomy',
  'CellType',
  'Chemical',
  'DevStage',
  'Disease',
  'GO_Term',
  'Gene',
  'Pathway',
  'Phenotype',
  'Phenotype_Quality',
  'Protein',
  'SequenceFeature',
  'Species'},
 {'Anatomy_Anatomy',
  'Anatomy_Species',
  'CellType_Anatomy',
  'CellType_CellType',
  'CellType_GO_Term',
  'Chemical_Chemical',
  'Chemical_Disease',
  'Chemical_GO_Term',
  'Chemical_Gene',
  'Chemical_Pathway',
  'DevStage_DevStage',
  'Disease_Disease',
  'Disease_GO_Term',
  'Disease_Pathway',
  'Disease_Phenotype',
  'GO_Term_Anatomy',
  'GO_Term_GO_Term',
  'GO_Term_Gene',
  'GO_Term_Pathway',
  'Gene_Gene',
  'Gene_Pathway',
  'Phenotype_Gene',
  'Phenotype_Phenotype',
  'Phenotype_Quality_Phenotype_Quality',
  'Protein_Chemical',
  'Protein_GO_Term',
  'Protein_Protein',
  'SequenceFeature_Protein',
  'SequenceFea

In [40]:
# # See all unique prefixes present
# all_prefixes = pd.concat([
#     df["subject_prefix"], 
#     df["object_prefix"]
# ]).value_counts()
# print(all_prefixes)

In [41]:
# all_prefixes.to_csv('prefix_display.txt')

In [50]:
import pandas as pd
import re

df = pd.read_csv(
    "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/pheknowlator/PheKnowLator_v1_Full_BioKG_NoDisjointness_Closed_ELK_Triples_Labels.txt",
    sep="\t", header=None, names=["subject", "object"]
)

# URI → clean ID
def parse_uri(uri):
    uri = str(uri)
    if "obolibrary.org/obo/" in uri:
        raw = uri.split("/obo/")[-1]          # e.g. DOID_5041, GO_0002537
        prefix = raw.split("_")[0]
        return raw.replace("_", ":"), prefix   # DOID:5041, GO, HP, CHEBI, etc.
    elif "uniprot.org/geneid/" in uri:
        return "NCBI:" + uri.split("/geneid/")[-1], "Gene"
    else:
        raw = uri.split("/")[-1].split("#")[-1]
        return raw, "Unknown"

# Map ontology prefix → node type
prefix_to_type = {
    "DOID": "Disease",
    "HP":   "Phenotype",
    "GO":   "GO_Term",
    "CHEBI":"Chemical",
    "REACTO":"Pathway",
    "PR":   "Protein",
    "CL":   "CellType",
    "UBERON":"Anatomy",
    "Gene": "Gene",
}

def get_type(prefix):
    return prefix_to_type.get(prefix, "Other")

df["subject_id"], df["subject_prefix"] = zip(*df["subject"].map(parse_uri))
df["object_id"],  df["object_prefix"]  = zip(*df["object"].map(parse_uri))
df["subject_type"] = df["subject_prefix"].map(get_type)
df["object_type"]  = df["object_prefix"].map(get_type)

# No relation column — infer from type pair
type_pair_to_relation = {
    ("Disease",  "GO_Term"):   "disease_associated_with_go",
    ("Chemical", "Disease"):   "chemical_associated_with_disease",
    ("Chemical", "Gene"):      "chemical_interacts_with_gene",
    ("GO_Term",  "GO_Term"):   "go_subclass_of",
    ("Gene",     "Gene"):      "gene_interacts_with_gene",
    ("Phenotype","Gene"):      "phenotype_associated_with_gene",
    ("Phenotype","Disease"):   "phenotype_associated_with_disease",
    ("GO_Term",  "Gene"):      "go_associated_with_gene",
}

df["relation"] = df.apply(
    lambda r: type_pair_to_relation.get(
        (r["subject_type"], r["object_type"]), "related_to"
    ), axis=1
)

# Final edge table
edges = df[[
    "subject_id", "relation", "object_id",
    "subject_type", "object_type"
]].rename(columns={
    "subject_id":   "Head",
    "object_id":    "Tail",
    "subject_type": "Head_type",
    "object_type":  "Tail_type",
    "relation":  "Relation"
})
edges["source"] = "PheKnowLator"

print(edges.shape)
print(edges.head(10))
# print(edges[["head_type","tail_type","relation"]].value_counts().head(20))
edges

(4283686, 6)
          Head                          Relation         Tail  Head_type  \
0    DOID:5041        disease_associated_with_go   GO:0002537    Disease   
1    DOID:1588        disease_associated_with_go   GO:0070269    Disease   
2   GO:0035377                    go_subclass_of   GO:0006833    GO_Term   
3  CHEBI:35455  chemical_associated_with_disease    DOID:2799   Chemical   
4  CHEBI:15930      chemical_interacts_with_gene   NCBI:10640   Chemical   
5   GO:0046982           go_associated_with_gene   NCBI:51085    GO_Term   
6    NCBI:5733          gene_interacts_with_gene   NCBI:10563       Gene   
7    NCBI:1778          gene_interacts_with_gene   NCBI:23047       Gene   
8  CHEBI:10782  chemical_associated_with_disease     DOID:813   Chemical   
9   HP:0000028    phenotype_associated_with_gene  NCBI:148789  Phenotype   

  Tail_type        source  
0   GO_Term  PheKnowLator  
1   GO_Term  PheKnowLator  
2   GO_Term  PheKnowLator  
3   Disease  PheKnowLator  
4      Gen

,Head,Relation,Tail,Head_type,Tail_type,source
0,DOID:5041,disease_associated_with_go,GO:0002537,Disease,GO_Term,PheKnowLator
1,DOID:1588,disease_associated_with_go,GO:0070269,Disease,GO_Term,PheKnowLator
2,GO:0035377,go_subclass_of,GO:0006833,GO_Term,GO_Term,PheKnowLator
3,CHEBI:35455,chemical_associated_with_disease,DOID:2799,Chemical,Disease,PheKnowLator
4,CHEBI:15930,chemical_interacts_with_gene,NCBI:10640,Chemical,Gene,PheKnowLator
...,...,...,...,...,...,...
4283681,DOID:8442,disease_associated_with_go,GO:0006749,Disease,GO_Term,PheKnowLator
4283682,GO:0030553,go_subclass_of,GO:0030553,GO_Term,GO_Term,PheKnowLator
4283683,DOID:10933,disease_associated_with_go,GO:1903188,Disease,GO_Term,PheKnowLator
4283684,DOID:12336,disease_associated_with_go,GO:0046500,Disease,GO_Term,PheKnowLator


In [51]:
edges['Relation'] = edges['Head_type'] + '_' + edges['Tail_type']
set(edges['Head_type']), set(edges['Tail_type']), set(edges['Relation'])

({'Anatomy',
  'CellType',
  'Chemical',
  'Disease',
  'GO_Term',
  'Gene',
  'Other',
  'Phenotype',
  'Protein'},
 {'Anatomy',
  'CellType',
  'Chemical',
  'Disease',
  'GO_Term',
  'Gene',
  'Other',
  'Phenotype',
  'Protein'},
 {'Anatomy_Anatomy',
  'Anatomy_Other',
  'CellType_CellType',
  'CellType_GO_Term',
  'CellType_Other',
  'Chemical_Chemical',
  'Chemical_Disease',
  'Chemical_GO_Term',
  'Chemical_Gene',
  'Chemical_Other',
  'Disease_Disease',
  'Disease_GO_Term',
  'Disease_Other',
  'Disease_Phenotype',
  'GO_Term_GO_Term',
  'GO_Term_Gene',
  'GO_Term_Other',
  'Gene_Gene',
  'Gene_Other',
  'Other_Anatomy',
  'Other_CellType',
  'Other_Chemical',
  'Other_Disease',
  'Other_GO_Term',
  'Other_Other',
  'Other_Protein',
  'Phenotype_Gene',
  'Phenotype_Other',
  'Phenotype_Phenotype',
  'Protein_Chemical',
  'Protein_GO_Term',
  'Protein_Other',
  'Protein_Protein'})

In [52]:
keep_relations = {
    'Anatomy_Anatomy', 'CellType_CellType', 'CellType_GO_Term',
    'Chemical_Chemical', 'Chemical_Disease', 'Chemical_GO_Term',
    'Chemical_Gene', 'Disease_Disease', 'Disease_GO_Term',
    'Disease_Phenotype', 'GO_Term_GO_Term', 'GO_Term_Gene',
    'Gene_Gene', 'Phenotype_Gene', 'Phenotype_Phenotype',
    'Protein_Chemical', 'Protein_GO_Term', 'Protein_Protein'
}

import os
out_dir = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/pheknowlator/semi_processed"
os.makedirs(out_dir, exist_ok=True)

filtered = edges[edges["Relation"].isin(keep_relations)]

for rel, grp in filtered.groupby("Relation"):
    grp.to_csv(f"{out_dir}/{rel}.csv", index=False)
    print(f"{rel}: {len(grp)} edges")

print(f"\nTotal edges saved: {len(filtered)}")

Anatomy_Anatomy: 8489 edges
CellType_CellType: 924 edges
CellType_GO_Term: 4 edges
Chemical_Chemical: 200531 edges
Chemical_Disease: 756419 edges
Chemical_GO_Term: 13700 edges
Chemical_Gene: 243902 edges
Disease_Disease: 2573 edges
Disease_GO_Term: 1429336 edges
Disease_Phenotype: 64105 edges
GO_Term_GO_Term: 95169 edges
GO_Term_Gene: 612393 edges
Gene_Gene: 411628 edges
Phenotype_Gene: 150653 edges
Phenotype_Phenotype: 27321 edges
Protein_Chemical: 5 edges
Protein_GO_Term: 1 edges
Protein_Protein: 600 edges

Total edges saved: 4017753


# Processing 

In [15]:
Anatomy_Anatomy = pd.read_csv(f'{BASE_PATH}pheknowlator/semi_processed/Anatomy_Anatomy.csv')
Anatomy_Anatomy['Head_detail_name'] = Anatomy_Anatomy['Head'].map(UBERON_dict)
Anatomy_Anatomy = Anatomy_Anatomy[~Anatomy_Anatomy['Head'].isna()]
Anatomy_Anatomy['Tail_detail_name'] = Anatomy_Anatomy['Tail'].map(UBERON_dict)
Anatomy_Anatomy = Anatomy_Anatomy[~Anatomy_Anatomy['Tail'].isna()]
Anatomy_Anatomy['Head_ID_IS'] = 'UBERON_ID'
Anatomy_Anatomy['Tail_ID_IS'] = 'UBERON_ID'
Anatomy_Anatomy['Relation']   = Anatomy_Anatomy['Head_type'] + '_' + Anatomy_Anatomy['Tail_type']
Anatomy_Anatomy = select_cols(Anatomy_Anatomy)
Anatomy_Anatomy['KG_source'] = 'pheknowlator'
save(Anatomy_Anatomy, 'Anatomy_Anatomy.csv')
Anatomy_Anatomy

Saved 8,489 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Anatomy_Anatomy.csv


,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,KG_source
0,UBERON:0005043,Anatomy_Anatomy,UBERON:0010314,Anatomy,Anatomy,mucosa of nasolacrimal duct,structure with developmental contribution from...,UBERON_ID,UBERON_ID,pheknowlator
1,UBERON:0010141,Anatomy_Anatomy,UBERON:0005295,Anatomy,Anatomy,primitive sex cord of indifferent gonad,sex cord,UBERON_ID,UBERON_ID,pheknowlator
2,UBERON:0002349,Anatomy_Anatomy,UBERON:0018260,Anatomy,Anatomy,myocardium,layer of muscle tissue,UBERON_ID,UBERON_ID,pheknowlator
3,UBERON:0018118,Anatomy_Anatomy,UBERON:0018114,Anatomy,Anatomy,right renal cortex interstitium,right kidney interstitium,UBERON_ID,UBERON_ID,pheknowlator
4,UBERON:0013704,Anatomy_Anatomy,UBERON:0004111,Anatomy,Anatomy,notochordal canal,anatomical conduit,UBERON_ID,UBERON_ID,pheknowlator
...,...,...,...,...,...,...,...,...,...,...
8484,UBERON:0004792,Anatomy_Anatomy,UBERON:0004795,Anatomy,Anatomy,secretion of endocrine pancreas,pancreas secretion,UBERON_ID,UBERON_ID,pheknowlator
8485,UBERON:0001742,Anatomy_Anatomy,UBERON:0001739,Anatomy,Anatomy,epiglottic cartilage,laryngeal cartilage,UBERON_ID,UBERON_ID,pheknowlator
8486,UBERON:0009505,Anatomy_Anatomy,UBERON:0010314,Anatomy,Anatomy,mesenchyme of trachea,structure with developmental contribution from...,UBERON_ID,UBERON_ID,pheknowlator
8487,UBERON:0015090,Anatomy_Anatomy,UBERON:0015049,Anatomy,Anatomy,distal carpal bone 3 endochondral element,carpus endochondral element,UBERON_ID,UBERON_ID,pheknowlator


In [16]:
CellType_CellType = pd.read_csv(f'{BASE_PATH}pheknowlator/semi_processed/CellType_CellType.csv')
CellType_CellType['Head_detail_name'] = CellType_CellType['Head'].map(CellOnto_dict)
CellType_CellType = CellType_CellType[~CellType_CellType['Head'].isna()]
CellType_CellType['Tail_detail_name'] = CellType_CellType['Tail'].map(CellOnto_dict)
CellType_CellType = CellType_CellType[~CellType_CellType['Tail'].isna()]
CellType_CellType['Head_ID_IS'] = 'OBO_ID'
CellType_CellType['Tail_ID_IS'] = 'OBO_ID'
CellType_CellType['Relation']   = CellType_CellType['Head_type'] + '_' + CellType_CellType['Tail_type']
CellType_CellType = select_cols(CellType_CellType)
CellType_CellType['KG_source'] = 'pheknowlator'
save(CellType_CellType, 'CellType_CellType.csv')
CellType_CellType

Saved 924 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/CellType_CellType.csv


,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,KG_source
0,CL:0002572,CellType_CellType,CL:0001035,CellType,CellType,vertebral mesenchymal stem cell,bone cell,OBO_ID,OBO_ID,pheknowlator
1,CL:1001586,CellType_CellType,CL:0002327,CellType,CellType,mammary gland glandular cell,mammary gland epithelial cell,OBO_ID,OBO_ID,pheknowlator
2,CL:0002260,CellType_CellType,CL:0000150,CellType,CellType,epithelial cell of parathyroid gland,glandular secretory epithelial cell,OBO_ID,OBO_ID,pheknowlator
3,CL:0002339,CellType_CellType,CL:0002341,CellType,CellType,prostate stem cell,basal cell of prostate epithelium,OBO_ID,OBO_ID,pheknowlator
4,CL:0000766,CellType_CellType,CL:0000763,CellType,CellType,myeloid leukocyte,myeloid cell,OBO_ID,OBO_ID,pheknowlator
...,...,...,...,...,...,...,...,...,...,...
919,CL:2000014,CellType_CellType,CL:0002620,CellType,CellType,fibroblast of upper leg skin,skin fibroblast,OBO_ID,OBO_ID,pheknowlator
920,CL:1001005,CellType_CellType,CL:1000746,CellType,CellType,glomerular capillary endothelial cell,glomerular cell,OBO_ID,OBO_ID,pheknowlator
921,CL:0001023,CellType_CellType,CL:0001030,CellType,CellType,"Kit-positive, CD34-positive common myeloid pro...",CD117-positive common myeloid progenitor OR CD...,OBO_ID,OBO_ID,pheknowlator
922,CL:0002236,CellType_CellType,CL:0002341,CellType,CellType,basal epithelial cell of prostatic duct,basal cell of prostate epithelium,OBO_ID,OBO_ID,pheknowlator


In [17]:
# # CellType_GO_Term: 4 edges

# CellType_GeneOnto = pd.read_csv(f'{BASE_PATH}pheknowlator/semi_processed/CellType_GO_Term.csv')

# CellType_GeneOnto['Head_detail_name'] = CellType_GeneOnto['Head'].map(CellOnto_dict)
# CellType_GeneOnto = CellType_GeneOnto[~CellType_GeneOnto['Head'].isna()]
# CellType_GeneOnto['Tail_detail_name'] = CellType_GeneOnto['Tail'].map(GO_dict)
# CellType_GeneOnto = CellType_GeneOnto[~CellType_GeneOnto['Tail'].isna()]
# CellType_GeneOnto['Tail_type'] = CellType_GeneOnto['Tail'].map(GO_namespace_dict)

# CellType_GeneOnto['Head_ID_IS'] = 'OBO_ID'
# CellType_GeneOnto['Tail_ID_IS'] = 'Quick_GO'
# CellType_GeneOnto['Relation']   = CellType_GeneOnto['Head_type'] + '_' + CellType_GeneOnto['Tail_type']
# CellType_GeneOnto = select_cols(CellType_GeneOnto)
# CellType_GeneOnto['KG_source'] = 'pheknowlator'
# save(CellType_GeneOnto, 'CellType_Cell.csv')

# CellType_GeneOnto

In [18]:
Chemical_Chemical = pd.read_csv(f'{BASE_PATH}pheknowlator/semi_processed/Chemical_Chemical.csv')

Chemical_Chemical['Head_ID'] = Chemical_Chemical['Head'].map(Chebi2PC_dict)
Chemical_Chemical['Tail_ID'] = Chemical_Chemical['Tail'].map(Chebi2PC_dict)
Chemical_Chemical = Chemical_Chemical.rename(columns={
    "Head": "Head_ID",
    "Head_ID": "Head",
    "Tail": "Tail_ID",
    "Tail_ID": "Tail"
})
Chemical_Chemical = Chemical_Chemical[~Chemical_Chemical['Head'].isna()]
Chemical_Chemical = Chemical_Chemical[~Chemical_Chemical['Tail'].isna()]
Chemical_Chemical['Head'] = Chemical_Chemical['Head'].astype(str).str.rstrip('.0')
Chemical_Chemical['Tail'] = Chemical_Chemical['Tail'].astype(str).str.rstrip('.0')

Chemical_Chemical
Chemical_Chemical['Head_IUPAC'] = Chemical_Chemical['Head'].map(Pubchem_IUPAC_CID_Dict)
Chemical_Chemical['Head_Smiles'] = Chemical_Chemical['Head'].map(Pubchem_CID_Smile_Dict)
Chemical_Chemical['Tail_IUPAC'] = Chemical_Chemical['Tail'].map(Pubchem_IUPAC_CID_Dict)
Chemical_Chemical['Tail_Smiles'] = Chemical_Chemical['Tail'].map(Pubchem_CID_Smile_Dict)

Chemical_Chemical['Head_ID_IS'] = 'Pubchem'
Chemical_Chemical['Tail_ID_IS'] = 'Pubchem'
Chemical_Chemical['Head_Type'] = 'ChemicalEntity'
Chemical_Chemical['Tail_Type'] = 'ChemicalEntity'
Chemical_Chemical['Relation']   = Chemical_Chemical['Head_type'] + '_' + Chemical_Chemical['Tail_type']
Chemical_Chemical = select_cols(Chemical_Chemical)
Chemical_Chemical['KG_source'] = 'pheknowlator'
save(Chemical_Chemical, 'Chemical_Chemical.csv')
Chemical_Chemical

Saved 14,698 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Chemical_Chemical.csv


,Head,Relation,Tail,Head_type,Tail_type,Head_IUPAC,Tail_IUPAC,Head_Smiles,Tail_Smiles,Head_ID_IS,Tail_ID_IS,KG_source
7,2805684,Chemical_Chemical,10176084,Chemical,Chemical,"3-[1-(2,4-dichlorophenyl)sulfonylpyrrolidin-2-...","1,2,3,4-tetrahydropyridine",C1CC(N(C1)S(=O)(=O)C2=C(C=C(C=C2)Cl)Cl)C3=CN=C...,C1CC=CNC1,Pubchem,Pubchem,pheknowlator
25,122164832,Chemical_Chemical,445638,Chemical,Chemical,[(2R)-2-[(Z)-hexadec-9-enoyl]oxy-3-hydroxyprop...,(Z)-hexadec-9-enoic acid,CCCCCC/C=C\CCCCCCCC(=O)O[C@H](CO)COP(=O)([O-])...,CCCCCC/C=C\CCCCCCCC(=O)O,Pubchem,Pubchem,pheknowlator
33,3083575,Chemical_Chemical,3083575,Chemical,Chemical,"2,8-dihydroxy-1-methoxy-3-methylanthracene-9,1...","2,8-dihydroxy-1-methoxy-3-methylanthracene-9,1...",CC1=CC2=C(C(=C1O)OC)C(=O)C3=C(C2=O)C=CC=C3O,CC1=CC2=C(C(=C1O)OC)C(=O)C3=C(C2=O)C=CC=C3O,Pubchem,Pubchem,pheknowlator
59,60184949,Chemical_Chemical,44237242,Chemical,Chemical,"(1S,5R)-3-(2-fluorophenyl)sulfonyl-7-[4-(3-met...","2-(hydroxymethyl)piperidin-1-ium-3,4,5-triol",COC1=CC=CC(=C1)C2=CC=C(C=C2)C3[C@H]4CN(C[C@@H]...,C1C(C(C(C([NH2+]1)CO)O)O)O,Pubchem,Pubchem,pheknowlator
60,50986185,Chemical_Chemical,139087999,Chemical,Chemical,"(1S,4R,4aS)-1,6-dimethyl-4-propan-2-yl-1,2,3,4...","1-[(1R,3aS,6R,7R,7aS)-6-hydroxy-4-methylidene-...",C[C@H]1CC[C@@H]([C@@H]2C1=CCC(=C2)C)C(C)C,CC(C)[C@H]1[C@@H](CC(=C)[C@@H]2[C@@H]1[C@@H](C...,Pubchem,Pubchem,pheknowlator
...,...,...,...,...,...,...,...,...,...,...,...,...
200452,11914,Chemical_Chemical,1292,Chemical,Chemical,(2R)-2-hydroxy-2-phenylacetic acid,2-hydroxy-2-phenylacetic acid,C1=CC=C(C=C1)[C@H](C(=O)O)O,C1=CC=C(C=C1)C(C(=O)O)O,Pubchem,Pubchem,pheknowlator
200508,6018498,Chemical_Chemical,44237242,Chemical,Chemical,[4-[(Z)-(thiophene-2-carbonylhydrazinylidene)m...,"2-(hydroxymethyl)piperidin-1-ium-3,4,5-triol",CC1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)/C=N\NC(=O)C3...,C1C(C(C(C([NH2+]1)CO)O)O)O,Pubchem,Pubchem,pheknowlator
200527,10241693,Chemical_Chemical,635,Chemical,Chemical,4-[[tert-butyl(hydroxy)amino]methylidene]cyclo...,"(3,5-dihydroxyoxolan-2-yl)methyl dihydrogen ph...",CC(C)(C)N(C=C1C=CC(=O)C=C1)O,C1C(C(OC1O)COP(=O)(O)O)O,Pubchem,Pubchem,pheknowlator
200529,440881,Chemical_Chemical,16069985,Chemical,Chemical,"[(2R,3R,4S,5S,6R)-5-acetamido-3,4-dihydroxy-6-...","[(3R,4S,5S,6R)-5-acetamido-3,4-dihydroxy-6-met...",C[C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)OP(=O)(O)O...,C[C@@H]1[C@H]([C@@H]([C@H](C(O1)OP(=O)(O)OP(=O...,Pubchem,Pubchem,pheknowlator


In [19]:
### 

In [20]:
Chemical_Disease = pd.read_csv(
    f'{BASE_PATH}pheknowlator/semi_processed/Chemical_Disease.csv'
)

Chemical_Disease['Head_ID'] = Chemical_Disease['Head'].map(Chebi2PC_dict)
Chemical_Disease['Tail_detail_name'] = Chemical_Disease['Tail'].map(DOID_disname_dict)

Chemical_Disease = Chemical_Disease.rename(columns={
    "Head": "Head_ID",
    "Head_ID": "Head",
    
})

Chemical_Disease = Chemical_Disease[~Chemical_Disease['Head'].isna()]
Chemical_Disease = Chemical_Disease[~Chemical_Disease['Tail_detail_name'].isna()]

Chemical_Disease['Head'] = (
    Chemical_Disease['Head']
    .astype(str)
    .str.rstrip('.0')
)

Chemical_Disease['Head_IUPAC'] = Chemical_Disease['Head'].map(Pubchem_IUPAC_CID_Dict)
Chemical_Disease['Head_Smiles'] = Chemical_Disease['Head'].map(Pubchem_CID_Smile_Dict)

Chemical_Disease['Head_ID_IS'] = 'Pubchem'
Chemical_Disease['Tail_ID_IS'] = 'DOID'

Chemical_Disease['Head_Type'] = 'ChemicalEntity'
Chemical_Disease['Tail_Type'] = 'Disease'

Chemical_Disease['Relation'] = (
    Chemical_Disease['Head_type'] + '_' +
    Chemical_Disease['Tail_type']
)

Chemical_Disease = select_cols(Chemical_Disease)

Chemical_Disease['KG_source'] = 'pheknowlator'

save(Chemical_Disease, 'Chemical_Disease.csv')

Chemical_Disease

Saved 659,986 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Chemical_Disease.csv


,Head,Relation,Tail,Head_type,Tail_type,Head_IUPAC,Head_Smiles,Tail_detail_name,Head_ID_IS,Tail_ID_IS,KG_source
0,1108,Chemical_Disease,DOID:2799,Chemical,Disease,"2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17-hexade...",C1CCC2C(C1)CCC3C2CCC4C3CCC4,bronchiolitis obliterans,Pubchem,DOID,pheknowlator
1,369312,Chemical_Disease,DOID:813,Chemical,Disease,[(4S)-4-prop-1-en-2-ylcyclohexen-1-yl]methanol,CC(=C)[C@H]1CCC(=CC1)CO,septic arthritis,Pubchem,DOID,pheknowlator
3,37439,Chemical_Disease,DOID:687,Chemical,Disease,"(3S,5S)-1,3,4,5-tetrahydroxycyclohexane-1-carb...",C1[C@@H](C([C@H](CC1(C(=O)O)O)O)O)O,hepatoblastoma,Pubchem,DOID,pheknowlator
4,338,Chemical_Disease,DOID:2352,Chemical,Disease,2-hydroxybenzoic acid,C1=CC=C(C(=C1)C(=O)O)O,hemochromatosis,Pubchem,DOID,pheknowlator
5,3036,Chemical_Disease,DOID:350,Chemical,Disease,"1-chloro-4-[2,2,2-trichloro-1-(4-chlorophenyl)...",C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)(Cl)Cl)Cl,mastocytosis,Pubchem,DOID,pheknowlator
...,...,...,...,...,...,...,...,...,...,...,...
756413,9152,Chemical_Disease,DOID:1240,Chemical,Disease,"pentacyclo[10.7.1.02,11.03,8.016,20]icosa-1(19...",C1=CC=C2C(=C1)C=CC3=C2C4=CC=CC5=C4C3=CC=C5,leukemia,Pubchem,DOID,pheknowlator
756415,5994,Chemical_Disease,DOID:1307,Chemical,Disease,"(8S,9S,10R,13S,14S,17S)-17-acetyl-10,13-dimeth...",CC(=O)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CCC4...,dementia,Pubchem,DOID,pheknowlator
756416,39042,Chemical_Disease,DOID:235,Chemical,Disease,2-[4-[2-[(4-chlorobenzoyl)amino]ethyl]phenoxy]...,CC(C)(C(=O)O)OC1=CC=C(C=C1)CCNC(=O)C2=CC=C(C=C...,colonic benign neoplasm,Pubchem,DOID,pheknowlator
756417,6434217,Chemical_Disease,DOID:4790,Chemical,Disease,"(3E)-1,7,7-trimethyl-3-[(4-methylphenyl)methyl...",CC1=CC=C(C=C1)/C=C/2\C3CCC(C2=O)(C3(C)C)C,medulloepithelioma,Pubchem,DOID,pheknowlator


In [21]:
Chemical_GO_Term = pd.read_csv(
    f'{BASE_PATH}pheknowlator/semi_processed/Chemical_GO_Term.csv'
)
Chemical_GO_Term
Chemical_GO_Term['Head_ID'] = Chemical_GO_Term['Head'].map(Chebi2PC_dict)
Chemical_GO_Term['Tail_detail_name'] = Chemical_GO_Term['Tail'].map(GO_dict)
Chemical_GO_Term = Chemical_GO_Term.rename(columns={
    "Head": "Head_ID",
    "Head_ID": "Head",
    
})
Chemical_GO_Term['Head'] = (
    Chemical_GO_Term['Head']
    .astype(str)
    .str.rstrip('.0')
)
Chemical_GO_Term = Chemical_GO_Term[~Chemical_GO_Term['Tail_detail_name'].isna()]

Chemical_GO_Term['Head_IUPAC'] = Chemical_GO_Term['Head'].map(Pubchem_IUPAC_CID_Dict)
Chemical_GO_Term['Head_Smiles'] = Chemical_GO_Term['Head'].map(Pubchem_CID_Smile_Dict)

Chemical_GO_Term['Head_Type'] = 'ChemicalEntity'
Chemical_GO_Term['Tail_type'] = Chemical_GO_Term['Tail'].map(GO_namespace_dict)

Chemical_GO_Term['Head_ID_IS'] = 'Pubchem_ID'
Chemical_GO_Term['Tail_ID_IS'] = 'Quick_GO'
Chemical_GO_Term['Relation']   = Chemical_GO_Term['Head_type'] + '_' + Chemical_GO_Term['Tail_type']
Chemical_GO_Term = Chemical_GO_Term[~Chemical_GO_Term['Relation'].isna()]
Chemical_GO_Term = select_cols(Chemical_GO_Term)
Chemical_GO_Term['KG_source'] = 'pheknowlator'
Chemical_GO_Term

# Biological Process
Chemical_BiologicalProcess = Chemical_GO_Term[
    Chemical_GO_Term['Tail_type'] == 'Biological Process'
]

save(Chemical_BiologicalProcess, 'Chemical_BiologicalProcess.csv')


# Cellular Component
Chemical_CellularComponent = Chemical_GO_Term[
    Chemical_GO_Term['Tail_type'] == 'Cellular Component'
]

save(Chemical_CellularComponent, 'Chemical_CellularComponent.csv')


# Molecular Function
Chemical_MolecularFunction = Chemical_GO_Term[
    Chemical_GO_Term['Tail_type'] == 'Molecular Function'
]

save(Chemical_MolecularFunction, 'Chemical_MolecularFunction.csv')

Saved 12,950 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Chemical_BiologicalProcess.csv
Saved 15 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Chemical_CellularComponent.csv
Saved 321 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Chemical_MolecularFunction.csv


In [22]:
Chemical_Gene = pd.read_csv(
    f'{BASE_PATH}pheknowlator/semi_processed/Chemical_Gene.csv'
)
Chemical_Gene

Chemical_Gene
Chemical_Gene['Head_ID'] = Chemical_Gene['Head'].map(Chebi2PC_dict)
Chemical_Gene['Tail'] = Chemical_Gene['Tail'].str.replace('NCBI:', '', regex=False)
# Chemical_Gene['Tail_ID'] = Chemical_Gene['Tail'].astype(int).map(NCBI_ALL_GENEIDname_dict)
Chemical_Gene['Tail_ID'] = (
    Chemical_Gene['Tail']
    .astype(str)
    .str.replace('NCBI:', '', regex=False)
    .astype(int)
    .map(NCBI_ALL_GENEIDname_dict)
)

Chemical_Gene = Chemical_Gene.rename(columns={
    "Head": "Head_ID",
    "Head_ID": "Head",
    "Tail": "Tail_ID",
    "Tail_ID": "Tail"
    
})

Chemical_Gene['Head_IUPAC'] = Chemical_Gene['Head'].map(Pubchem_IUPAC_CID_Dict)
Chemical_Gene['Head_Smiles'] = Chemical_Gene['Head'].map(Pubchem_CID_Smile_Dict)
Chemical_Gene['Tail_detail_name'] = Chemical_Gene['Tail'].map(NCBI_Synbol_GENEname_dict)
Chemical_Gene['Head'] = (
    Chemical_Gene['Head']
    .astype(str)
    .str.rstrip('.0')
)
Chemical_Gene = Chemical_Gene[
    Chemical_Gene['Head'].astype(str) != 'nan'
]
Chemical_Gene = Chemical_Gene[~Chemical_Gene['Tail_detail_name'].isna()]


Chemical_Gene['Head_Type'] = 'ChemicalEntity'
Chemical_Gene['Head_ID_IS'] = 'Pubchem_ID'
Chemical_Gene['Tail_ID_IS'] = 'NCBI_ID'
Chemical_Gene['Relation']   = Chemical_Gene['Head_type'] + '_' + Chemical_Gene['Tail_type']
Chemical_Gene = Chemical_Gene[~Chemical_Gene['Relation'].isna()]
Chemical_Gene = select_cols(Chemical_Gene)
Chemical_Gene['KG_source'] = 'pheknowlator'
Chemical_Gene
save(Chemical_Gene, 'Chemical_Gene.csv')

Saved 223,559 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Chemical_Gene.csv


In [23]:
Chemical_Gene

,Head,Relation,Tail,Head_type,Tail_type,Head_IUPAC,Head_Smiles,Tail_detail_name,Head_ID_IS,Tail_ID_IS,KG_source
0,2256,Chemical_Gene,EXOC5,Chemical,Gene,NaN,NaN,exocyst complex component 5,Pubchem_ID,NCBI_ID,pheknowlator
2,445639,Chemical_Gene,LPIN1,Chemical,Gene,NaN,NaN,lipin 1,Pubchem_ID,NCBI_ID,pheknowlator
3,3121,Chemical_Gene,ADCY2,Chemical,Gene,NaN,NaN,adenylate cyclase 2,Pubchem_ID,NCBI_ID,pheknowlator
4,31703,Chemical_Gene,LAMA5,Chemical,Gene,NaN,NaN,laminin subunit alpha 5,Pubchem_ID,NCBI_ID,pheknowlator
6,9064,Chemical_Gene,DEDD,Chemical,Gene,NaN,NaN,death effector domain containing,Pubchem_ID,NCBI_ID,pheknowlator
...,...,...,...,...,...,...,...,...,...,...,...
243895,5591,Chemical_Gene,TOP2A,Chemical,Gene,NaN,NaN,DNA topoisomerase II alpha,Pubchem_ID,NCBI_ID,pheknowlator
243897,6758,Chemical_Gene,LINC01698,Chemical,Gene,NaN,NaN,long intergenic non-protein coding RNA 1698,Pubchem_ID,NCBI_ID,pheknowlator
243899,5641,Chemical_Gene,TNFAIP3,Chemical,Gene,NaN,NaN,TNF alpha induced protein 3,Pubchem_ID,NCBI_ID,pheknowlator
243900,9823887,Chemical_Gene,MC1R,Chemical,Gene,NaN,NaN,melanocortin 1 receptor,Pubchem_ID,NCBI_ID,pheknowlator


In [34]:
# Disease_Disease = pd.read_csv(  #self loop  Discarded
#     f'{BASE_PATH}pheknowlator/semi_processed/Disease_Disease.csv'
# )

# # Disease_Disease['Head_detail_name'] = Disease_Disease['Head'].map(DOID_disname_dict)
# # Disease_Disease['Tail_detail_name'] = Disease_Disease['Tail'].map(DOID_disname_dict)

# # Disease_Disease = Disease_Disease[~Disease_Disease['Head_detail_name'].isna()]
# # Disease_Disease = Disease_Disease[~Disease_Disease['Tail_detail_name'].isna()]

# # Disease_Disease['Head_ID_IS'] = 'DOID'
# # Disease_Disease['Tail_ID_IS'] = 'DOID'

# # Disease_Disease['Head_Type'] = 'Disease'
# # Disease_Disease['Tail_Type'] = 'Disease'

# # Disease_Disease['Relation'] = (
# #     Disease_Disease['Head_type'] + '_' +
# #     Disease_Disease['Tail_type']
# # )

# # Disease_Disease = select_cols(Disease_Disease)


# # save(Disease_Disease, 'Disease_Disease.csv')
# Disease_Disease

,Head,Relation,Tail,Head_type,Tail_type,source
0,DOID:61,Disease_Disease,DOID:61,Disease,Disease,PheKnowLator
1,DOID:0050948,Disease_Disease,DOID:0050948,Disease,Disease,PheKnowLator
2,DOID:0080567,Disease_Disease,DOID:0080567,Disease,Disease,PheKnowLator
3,DOID:0060711,Disease_Disease,DOID:0060711,Disease,Disease,PheKnowLator
4,DOID:0070237,Disease_Disease,DOID:0070237,Disease,Disease,PheKnowLator
...,...,...,...,...,...,...
2568,DOID:0080419,Disease_Disease,DOID:0080419,Disease,Disease,PheKnowLator
2569,DOID:0110849,Disease_Disease,DOID:0110849,Disease,Disease,PheKnowLator
2570,DOID:0050700,Disease_Disease,DOID:0050700,Disease,Disease,PheKnowLator
2571,DOID:0110328,Disease_Disease,DOID:0110328,Disease,Disease,PheKnowLator


In [25]:
Disease_GO_Term = pd.read_csv(  
    f'{BASE_PATH}pheknowlator/semi_processed/Disease_GO_Term.csv'
)

Disease_GO_Term['Head_detail_name'] = Disease_GO_Term['Head'].map(DOID_disname_dict)
Disease_GO_Term['Tail_detail_name'] = Disease_GO_Term['Tail'].map(GO_dict)

Disease_GO_Term = Disease_GO_Term[~Disease_GO_Term['Head_detail_name'].isna()]
Disease_GO_Term = Disease_GO_Term[~Disease_GO_Term['Tail_detail_name'].isna()]


Disease_GO_Term['Head_ID_IS'] = 'DOID'
Disease_GO_Term['Tail_ID_IS'] = 'Quick_GO'

Disease_GO_Term['Tail_type'] = Disease_GO_Term['Tail'].map(GO_namespace_dict)

Disease_GO_Term['Relation'] = (
    Disease_GO_Term['Head_type'] + '_' +
    Disease_GO_Term['Tail_type']
)

Disease_GO_Term = select_cols(Disease_GO_Term)

Disease_GO_Term['KG_source'] = 'pheknowlator'


# Biological Process
Disease_BiologicalProcess = Disease_GO_Term[
    Disease_GO_Term['Tail_type'] == 'Biological Process'
]

save(Disease_BiologicalProcess, 'Disease_BiologicalProcess.csv')


# Cellular Component
Disease_CellularComponent = Disease_GO_Term[
    Disease_GO_Term['Tail_type'] == 'Cellular Component'
]

save(Disease_CellularComponent, 'Disease_CellularComponent.csv')


# Molecular Function
Disease_MolecularFunction = Disease_GO_Term[
    Disease_GO_Term['Tail_type'] == 'Molecular Function'
]
save(Disease_MolecularFunction, 'Disease_MolecularFunction.csv')


Saved 1,181,971 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Disease_BiologicalProcess.csv
Saved 80,482 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Disease_CellularComponent.csv
Saved 133,228 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Disease_MolecularFunction.csv


In [26]:
Disease_Phenotype = pd.read_csv(  
    f'{BASE_PATH}pheknowlator/semi_processed/Disease_Phenotype.csv'
)

Disease_Phenotype['Head_detail_name'] = Disease_Phenotype['Head'].map(DOID_disname_dict)
Disease_Phenotype['Tail_detail_name'] = Disease_Phenotype['Tail'].map(HPOphenotype_name_dict)
Disease_Phenotype = Disease_Phenotype[~Disease_Phenotype['Head_detail_name'].isna()]
Disease_Phenotype = Disease_Phenotype[~Disease_Phenotype['Tail_detail_name'].isna()]

Disease_Phenotype['Head_ID_IS'] = 'DOID'
Disease_Phenotype['Tail_ID_IS'] = 'HPO_ID'

Disease_Phenotype
Disease_Phenotype['Relation'] = (
    Disease_Phenotype['Head_type'] + '_' +
    Disease_Phenotype['Tail_type']
)

Disease_Phenotype = select_cols(Disease_Phenotype)
Disease_Phenotype['KG_source'] = 'pheknowlator'
Disease_Phenotype
save(Disease_Phenotype, 'Disease_Phenotype.csv')

Saved 63,825 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Disease_Phenotype.csv


In [27]:
GO_Term_GO_Term = pd.read_csv(  
    f'{BASE_PATH}pheknowlator/semi_processed/GO_Term_GO_Term.csv'
)
GO_Term_GO_Term['Head_detail_name'] = GO_Term_GO_Term['Head'].map(GO_dict)
GO_Term_GO_Term['Tail_detail_name'] = GO_Term_GO_Term['Tail'].map(GO_dict)

GO_Term_GO_Term = GO_Term_GO_Term[~GO_Term_GO_Term['Head_detail_name'].isna()]
GO_Term_GO_Term = GO_Term_GO_Term[~GO_Term_GO_Term['Tail_detail_name'].isna()]

GO_Term_GO_Term['Head_ID_IS'] = 'Quick_GO'
GO_Term_GO_Term['Tail_ID_IS'] = 'Quick_GO'

GO_Term_GO_Term['Head_type'] = GO_Term_GO_Term['Head'].map(GO_namespace_dict)

GO_Term_GO_Term['Tail_type'] = GO_Term_GO_Term['Tail'].map(GO_namespace_dict)

GO_Term_GO_Term['Relation'] = (
    GO_Term_GO_Term['Head_type'] + '_' +
    GO_Term_GO_Term['Tail_type']
)

GO_Term_GO_Term = select_cols(GO_Term_GO_Term)

GO_Term_GO_Term['KG_source'] = 'pheknowlator'

GO_Term_GO_Term

# Biological Process
BiologicalProcess_BiologicalProcess = GO_Term_GO_Term[
    GO_Term_GO_Term['Relation'] == 'Biological Process_Biological Process'
]
save(BiologicalProcess_BiologicalProcess, 'BiologicalProcess_BiologicalProcess.csv')


# Cellular Component
CellularComponent_CellularComponent = GO_Term_GO_Term[
    GO_Term_GO_Term['Relation'] == 'Cellular Component_Cellular Component'
]
save(CellularComponent_CellularComponent, 'CellularComponent_CellularComponent.csv')


# Molecular Function
MolecularFunction_MolecularFunction = GO_Term_GO_Term[
    GO_Term_GO_Term['Relation'] == 'Molecular Function_Molecular Function'
]
save(MolecularFunction_MolecularFunction, 'MolecularFunction_MolecularFunction.csv')

Saved 64,889 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/BiologicalProcess_BiologicalProcess.csv
Saved 7,480 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/CellularComponent_CellularComponent.csv
Saved 16,859 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/MolecularFunction_MolecularFunction.csv


In [28]:
set(GO_Term_GO_Term['Relation'])

{'Biological Process_Biological Process',
 'Cellular Component_Cellular Component',
 'Molecular Function_Molecular Function'}

In [29]:
GO_Term_Gene = pd.read_csv(  
    f'{BASE_PATH}pheknowlator/semi_processed/GO_Term_Gene.csv'
)

GO_Term_Gene['Head_detail_name'] = GO_Term_Gene['Head'].map(GO_dict)
GO_Term_Gene['Head_ID_IS'] = 'Quick_GO'
GO_Term_Gene['Head_type'] = GO_Term_Gene['Head'].map(GO_namespace_dict)

GO_Term_Gene['Tail'] = GO_Term_Gene['Tail'].str.replace('NCBI:', '', regex=False)
GO_Term_Gene['Tail_ID'] = (
    GO_Term_Gene['Tail']
    .astype(str)
    .str.replace('NCBI:', '', regex=False)
    .astype(int)
    .map(NCBI_ALL_GENEIDname_dict)
)

GO_Term_Gene = GO_Term_Gene.rename(columns={
    "Tail": "Tail_ID",
    "Tail_ID": "Tail"
    
})
GO_Term_Gene['Tail_detail_name'] = GO_Term_Gene['Tail'].map(NCBI_Synbol_GENEname_dict)
# GO_Term_Gene = GO_Term_Gene[~GO_Term_Gene['Tail_detail_name'].isna()]

GO_Term_Gene['Relation'] = (
    GO_Term_Gene['Head_type'] + '_' +
    GO_Term_Gene['Tail_type']
)

GO_Term_Gene = select_cols(GO_Term_Gene)
GO_Term_Gene['KG_source'] = 'pheknowlator'
GO_Term_Gene

# Biological Process
BiologicalProcess = GO_Term_Gene[
    GO_Term_Gene['Head_type'] == 'Biological Process'
]
save(BiologicalProcess, 'BiologicalProcess_Gene.csv')


# Cellular Component
CellularComponent = GO_Term_Gene[
    GO_Term_Gene['Head_type'] == 'Cellular Component'
]
save(CellularComponent, 'CellularComponent_Gene.csv')


# Molecular Function
MolecularFunction = GO_Term_Gene[
    GO_Term_Gene['Head_type'] == 'Molecular Function'
]
save(MolecularFunction, 'MolecularFunction_Gene.csv')

Saved 278,928 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/BiologicalProcess_Gene.csv
Saved 172,407 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/CellularComponent_Gene.csv
Saved 145,233 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/MolecularFunction_Gene.csv


In [30]:
Gene_Gene = pd.read_csv(  
    f'{BASE_PATH}pheknowlator/semi_processed/Gene_Gene.csv'
)
Gene_Gene

Gene_Gene['Head'] = Gene_Gene['Head'].str.replace('NCBI:', '', regex=False)
Gene_Gene['Head_ID'] = (
    Gene_Gene['Head']
    .astype(str)
    .str.replace('NCBI:', '', regex=False)
    .astype(int)
    .map(NCBI_ALL_GENEIDname_dict)
)


Gene_Gene['Tail'] = Gene_Gene['Tail'].str.replace('NCBI:', '', regex=False)
Gene_Gene['Tail_ID'] = (
    Gene_Gene['Tail']
    .astype(str)
    .str.replace('NCBI:', '', regex=False)
    .astype(int)
    .map(NCBI_ALL_GENEIDname_dict)
)

Gene_Gene = Gene_Gene.rename(columns={
    "Head": "Head_ID",
    "Head_ID": "Head",
    "Tail": "Tail_ID",
    "Tail_ID": "Tail"
    
})
Gene_Gene['Head_detail_name'] = Gene_Gene['Head'].map(NCBI_Synbol_GENEname_dict)

Gene_Gene['Tail_detail_name'] = Gene_Gene['Tail'].map(NCBI_Synbol_GENEname_dict)
Gene_Gene = Gene_Gene[~Gene_Gene['Head_detail_name'].isna()]
Gene_Gene = Gene_Gene[~Gene_Gene['Tail_detail_name'].isna()]

Gene_Gene['Relation'] = (
    Gene_Gene['Head_type'] + '_' +
    Gene_Gene['Tail_type']
)

Gene_Gene = select_cols(Gene_Gene)
Gene_Gene['KG_source'] = 'pheknowlator'
Gene_Gene

save(Gene_Gene, 'Gene_Gene.csv')
Gene_Gene

Saved 410,698 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Gene_Gene.csv


,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,KG_source
0,PTGER3,Gene_Gene,CXCL13,Gene,Gene,prostaglandin E receptor 3,C-X-C motif chemokine ligand 13,pheknowlator
1,DYNC1H1,Gene_Gene,PDS5B,Gene,Gene,dynein cytoplasmic 1 heavy chain 1,PDS5 cohesin associated factor B,pheknowlator
2,SSTR3,Gene_Gene,BBS2,Gene,Gene,somatostatin receptor 3,Bardet-Biedl syndrome 2,pheknowlator
3,TERF1,Gene_Gene,H2AC6,Gene,Gene,telomeric repeat binding factor 1,H2A clustered histone 6,pheknowlator
4,NUCB1,Gene_Gene,FUCA2,Gene,Gene,nucleobindin 1,alpha-L-fucosidase 2,pheknowlator
...,...,...,...,...,...,...,...,...
411623,KLHL11,Gene_Gene,RPS27A,Gene,Gene,kelch like family member 11,ribosomal protein S27a,pheknowlator
411624,PES1,Gene_Gene,RPL23,Gene,Gene,pescadillo ribosomal biogenesis factor 1,ribosomal protein L23,pheknowlator
411625,FLT3,Gene_Gene,STAT3,Gene,Gene,fms related receptor tyrosine kinase 3,signal transducer and activator of transcripti...,pheknowlator
411626,KLHL9,Gene_Gene,ASB15,Gene,Gene,kelch like family member 9,ankyrin repeat and SOCS box containing 15,pheknowlator


In [31]:
Phenotype_Gene = pd.read_csv(  
    f'{BASE_PATH}pheknowlator/semi_processed/Phenotype_Gene.csv'
)

Phenotype_Gene['Head_detail_name'] = Phenotype_Gene['Head'].map(HPOphenotype_name_dict)
Phenotype_Gene['Tail'] = Phenotype_Gene['Tail'].str.replace('NCBI:', '', regex=False)
Phenotype_Gene['Tail_ID'] = (
    Phenotype_Gene['Tail']
    .astype(str)
    .str.replace('NCBI:', '', regex=False)
    .astype(int)
    .map(NCBI_ALL_GENEIDname_dict)
)
Phenotype_Gene = Phenotype_Gene.rename(columns={
    "Tail": "Tail_ID",
    "Tail_ID": "Tail"
})

Phenotype_Gene['Tail_detail_name'] = Phenotype_Gene['Tail'].map(NCBI_Synbol_GENEname_dict)
Phenotype_Gene = Phenotype_Gene[~Phenotype_Gene['Head_detail_name'].isna()]
Phenotype_Gene = Phenotype_Gene[~Phenotype_Gene['Tail_detail_name'].isna()]

Phenotype_Gene['Head_ID_IS'] = 'HPO_ID'
Phenotype_Gene['Tail_ID_IS'] = 'NCBI_ID'

Phenotype_Gene = select_cols(Phenotype_Gene)
Phenotype_Gene['KG_source'] = 'pheknowlator'
Phenotype_Gene
save(Phenotype_Gene, 'Phenotype_Gene.csv')
Phenotype_Gene

Saved 150,653 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Phenotype_Gene.csv


,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,KG_source
0,HP:0000028,Phenotype_Gene,B3GALNT2,Phenotype,Gene,Cryptorchidism,"beta-1,3-N-acetylgalactosaminyltransferase 2",HPO_ID,NCBI_ID,pheknowlator
1,HP:0002920,Phenotype_Gene,CDH23,Phenotype,Gene,Decreased circulating ACTH concentration,cadherin related 23,HPO_ID,NCBI_ID,pheknowlator
2,HP:0000726,Phenotype_Gene,TRNF,Phenotype,Gene,Dementia,tRNA-Phe,HPO_ID,NCBI_ID,pheknowlator
3,HP:0002516,Phenotype_Gene,MLH1,Phenotype,Gene,Increased intracranial pressure,mutL homolog 1,HPO_ID,NCBI_ID,pheknowlator
4,HP:0005144,Phenotype_Gene,SYNE1,Phenotype,Gene,Ventricular septal hypertrophy,spectrin repeat containing nuclear envelope pr...,HPO_ID,NCBI_ID,pheknowlator
...,...,...,...,...,...,...,...,...,...,...
150648,HP:0002716,Phenotype_Gene,IGH,Phenotype,Gene,Lymphadenopathy,immunoglobulin heavy locus,HPO_ID,NCBI_ID,pheknowlator
150649,HP:0030914,Phenotype_Gene,CLMP,Phenotype,Gene,Abnormal peristalsis,CXADR like membrane protein,HPO_ID,NCBI_ID,pheknowlator
150650,HP:0045074,Phenotype_Gene,RNU4ATAC,Phenotype,Gene,Thin eyebrow,"RNA, U4atac small nuclear",HPO_ID,NCBI_ID,pheknowlator
150651,HP:0000873,Phenotype_Gene,SOX3,Phenotype,Gene,Diabetes insipidus,SRY-box transcription factor 3,HPO_ID,NCBI_ID,pheknowlator


In [32]:
Phenotype_Phenotype = pd.read_csv(   # # Disease_Disease = pd.read_csv( 
    f'{BASE_PATH}pheknowlator/semi_processed/Phenotype_Phenotype.csv'
)

Phenotype_Phenotype['Head_detail_name'] = Phenotype_Phenotype['Head'].map(HPOphenotype_name_dict)
Phenotype_Phenotype['Tail_detail_name'] = Phenotype_Phenotype['Tail'].map(HPOphenotype_name_dict)


Phenotype_Phenotype = Phenotype_Phenotype[~Phenotype_Phenotype['Head_detail_name'].isna()]
Phenotype_Phenotype = Phenotype_Phenotype[~Phenotype_Phenotype['Tail_detail_name'].isna()]

Phenotype_Phenotype['Head_ID_IS'] = 'HPO_ID'
Phenotype_Phenotype['Tail_ID_IS'] = 'HPO_ID'

Phenotype_Phenotype = select_cols(Phenotype_Phenotype)
Phenotype_Phenotype['KG_source'] = 'pheknowlator'

save(Phenotype_Phenotype, 'Phenotype_Phenotype.csv')

Phenotype_Phenotype

Saved 27,321 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/pheknowlator/Phenotype_Phenotype.csv


,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,KG_source
0,HP:0011353,Phenotype_Phenotype,HP:0011353,Phenotype,Phenotype,Arterial intimal fibrosis,Arterial intimal fibrosis,HPO_ID,HPO_ID,pheknowlator
1,HP:0009327,Phenotype_Phenotype,HP:0009334,Phenotype,Phenotype,Ivory epiphysis of the middle phalanx of the 3...,Abnormality of the epiphysis of the middle pha...,HPO_ID,HPO_ID,pheknowlator
2,HP:0100089,Phenotype_Phenotype,HP:0010323,Phenotype,Phenotype,Abnormality of the epiphysis of the middle pha...,Abnormality of the epiphyses of the 2nd toe,HPO_ID,HPO_ID,pheknowlator
3,HP:0012600,Phenotype_Phenotype,HP:0012591,Phenotype,Phenotype,Abnormal urine chloride concentration,Abnormal urinary electrolyte concentration,HPO_ID,HPO_ID,pheknowlator
4,HP:0100902,Phenotype_Phenotype,HP:0100920,Phenotype,Phenotype,Sclerosis of the distal phalanx of the 4th finger,Sclerosis of 4th finger phalanx,HPO_ID,HPO_ID,pheknowlator
...,...,...,...,...,...,...,...,...,...,...
27316,HP:0003261,Phenotype_Phenotype,HP:0010702,Phenotype,Phenotype,Increased circulating IgA concentration,Increased circulating antibody concentration,HPO_ID,HPO_ID,pheknowlator
27317,HP:0002032,Phenotype_Phenotype,HP:0002032,Phenotype,Phenotype,Esophageal atresia,Esophageal atresia,HPO_ID,HPO_ID,pheknowlator
27318,HP:0002186,Phenotype_Phenotype,HP:0002186,Phenotype,Phenotype,Apraxia,Apraxia,HPO_ID,HPO_ID,pheknowlator
27319,HP:0040091,Phenotype_Phenotype,HP:0010722,Phenotype,Phenotype,Asymmetry of the size of ears,Asymmetry of the ears,HPO_ID,HPO_ID,pheknowlator


In [33]:
#